# Response generation

[Fine-tuning Llama 2 with LoRA](https://deci.ai/blog/fine-tune-llama-2-with-lora-for-question-answering/)

# Testing Data
testing different data structure

In [81]:
import os, torch, logging
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, HfArgumentParser, TrainingArguments, pipeline
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

from tqdm import tqdm
from flash_attn import flash_attn_func
import accelerate
from openai import OpenAI
import wandb
from huggingface_hub import login
import os

In [82]:
%load_ext dotenv
%dotenv

login(token=os.environ.get("HF_TOKEN", ""), add_to_git_credential=True)
client = OpenAI(
  api_key=os.environ.get("OPENAI_API_KEY"),
)

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (cache).
Your token has been saved to /home/user/.cache/huggingface/token
Login successful


## OpenAI API
丟整個 list 叫他改寫效果很差，一句一句來效果比較好。

In [83]:
completion = client.chat.completions.create(
  model="gpt-4-turbo",
  messages=[
    {"role": "system", "content": "rewrite with anger emotions"},
    {"role": "user", "content": "Say , Jim , how about going for a few beers after dinner ?"}
  ],
)

print(completion.choices[0].message.content)

Seriously, Jim, can't we just go grab some beers after dinner for once? Would it kill you to say yes?


## Dataset

Observe first 10% of data from better_daily_dialog

In [84]:
dd = load_dataset("benjaminbeilharz/better_daily_dialog", split="train[:10%]")

Get the first 100 utterance of the dataset and send to chatGPT

In [85]:
def example(content):
    return [
        {"role": "system", "content": "rewrite with anger emotions"},
        {"role": "user", "content": content}
    ]

In [95]:
def format_dialogue(content, response):
    return [
        {"role": "system", "content": "rewrite with anger emotions"},
        {"role": "user", "content": content},
        {"role": "assistant", "content": response},
    ]

dd_list = dd[0:100]['utterance']
chats = []

pbar = tqdm(total=len(dd_list))

for content in dd_list:
    completion = client.chat.completions.create(
        model="gpt-4-turbo",
        messages=example(content)
    )
    response = completion.choices[0].message.content
    chats.append(format_dialogue(content, response))
    pbar.update(1)

pbar.close()


for i, chat in enumerate(chats):
    print(f"chat{i+1} = {chat}")


100%|██████████| 100/100 [03:37<00:00,  2.18s/it]

chat1 = [{'role': 'system', 'content': 'rewrite with anger emotions'}, {'role': 'user', 'content': 'Say , Jim , how about going for a few beers after dinner ? '}, {'role': 'assistant', 'content': "Jim, seriously, how many times do I have to ask you? Let's grab some beers after dinner already!"}]
chat2 = [{'role': 'system', 'content': 'rewrite with anger emotions'}, {'role': 'user', 'content': ' You know that is tempting but is really not good for our fitness . '}, {'role': 'assistant', 'content': "I can't believe you're even suggesting that! It's so blatantly bad for our fitness goals!"}]
chat3 = [{'role': 'system', 'content': 'rewrite with anger emotions'}, {'role': 'user', 'content': ' What do you mean ? It will help us to relax . '}, {'role': 'assistant', 'content': "What do you mean it'll help us relax? How can you even say that with everything that's going on right now?"}]
chat4 = [{'role': 'system', 'content': 'rewrite with anger emotions'}, {'role': 'user', 'content': " Do you r

In [96]:
# chats[1]

In [97]:
# test_data = []
# for sublist in chats:
#     test_data.append(sublist[:2])

# test_data[1]

## Model
setup model

In [98]:
# Model and tokenizer names
base_model_name = "meta-llama/Llama-2-7b-chat-hf"

# Tokenizer
llama_tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "right"  # Fix for fp16

## Data processing
paking up and shits...

In [99]:
dataset = Dataset.from_dict({"chat": chats})
dataset = dataset.map(lambda x: {"formatted_chat": llama_tokenizer.apply_chat_template(x["chat"], tokenize=False, add_generation_prompt=False)})
print(dataset['formatted_chat'])

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

["<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\nSay , Jim , how about going for a few beers after dinner ? [/INST] Jim, seriously, how many times do I have to ask you? Let's grab some beers after dinner already! </s>", "<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\n You know that is tempting but is really not good for our fitness . [/INST] I can't believe you're even suggesting that! It's so blatantly bad for our fitness goals! </s>", "<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\n What do you mean ? It will help us to relax . [/INST] What do you mean it'll help us relax? How can you even say that with everything that's going on right now? </s>", "<s>[INST] <<SYS>>\nrewrite with anger emotions\n<</SYS>>\n\n Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ? [/INST] Are you kidding me? How can you even think that's a good idea after what happened last time? It just made us fat and act like complete f

In [100]:
dataset

Dataset({
    features: ['chat', 'formatted_chat'],
    num_rows: 100
})

### Saving them for later use
我不太確定能不能用 `train_test_split` 直接切一切，姑且先分開存反正很小。

In [101]:
dataset.to_json("dataset_v1.json")
# test_dataset.to_json("test_dataset_v1.json")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

64597

In [102]:
llama_tokenizer.chat_template

"{% if messages[0]['role'] == 'system' %}{% set loop_messages = messages[1:] %}{% set system_message = messages[0]['content'] %}{% else %}{% set loop_messages = messages %}{% set system_message = false %}{% endif %}{% for message in loop_messages %}{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}{{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}{% endif %}{% if loop.index0 == 0 and system_message != false %}{% set content = '<<SYS>>\\n' + system_message + '\\n<</SYS>>\\n\\n' + message['content'] %}{% else %}{% set content = message['content'] %}{% endif %}{% if message['role'] == 'user' %}{{ bos_token + '[INST] ' + content.strip() + ' [/INST]' }}{% elif message['role'] == 'assistant' %}{{ ' '  + content.strip() + ' ' + eos_token }}{% endif %}{% endfor %}"

In [103]:

llama_tokenizer.default_chat_template


No chat template is defined for this tokenizer - using the default template for the LlamaTokenizerFast class. If the default is not appropriate for your model, please set `tokenizer.chat_template` to an appropriate template. See https://huggingface.co/docs/transformers/main/chat_templating for more information.



"{% if messages[0]['role'] == 'system' %}{% set loop_messages = messages[1:] %}{% set system_message = messages[0]['content'] %}{% elif false == true and not '<<SYS>>' in messages[0]['content'] %}{% set loop_messages = messages %}{% set system_message = 'You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature.\\n\\nIf a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct. If you don\\'t know the answer to a question, please don\\'t share false information.' %}{% else %}{% set loop_messages = messages %}{% set system_message = false %}{% endif %}{% for message in loop_messages %}{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}{{ raise_exception('Conversation roles must